# Milestone 2 lab, notebook 3 of 8: When the API says no

## Where we are

| Notebook | What you do |
|---|---|
| 1. Ask only for what changed | Read the orders that changed, not all of them |
| 2. Remember where you stopped | Keep a watermark. See what happens when it moves too early |
| **3. When the API says no** (you are here) | Meet the rate limit and a server error. Wait, and try again |
| 4. Land it raw | Put every order in Postgres, exactly as it arrived |
| 5. Pull or push | A short lab: let the order system send the changes to you |
| 6. What you built has a name | Change data capture, and what this kind cannot see |
| 7. Counted twice | Run a few mornings. Watch the report go wrong |
| 8. The production version | The same logic as a tested program |

The pipeline now asks for less. But it still breaks the moment the order system says "not now". A real source says that often. This notebook makes the load wait, and try again.

In [1]:
import time

import httpx

from lab_helpers import api, read_orders, set_morning, show_doc

## Step 1: Pages, again

You have followed pages since milestone 1. Here is the word for it, and one detail worth knowing.

**Pagination**: an API returns rows in pages, and you ask again until it says it is done. There are two common ways to ask for "the next page":

| Way | It says | When data changes while you read |
|---|---|---|
| Offset | "Skip 50 rows, then give me 25" | A new row shifts everything. You read one row twice, or skip one |
| Cursor (this API) | "Give me the rows after this bookmark" | The bookmark stays correct |

## Step 2: The rate limit

A **rate limit** caps how many calls you may make in a period. This API allows 12 calls in 10 seconds. Make 13.

In [2]:
set_morning("2026-06-09")

for call in range(1, 14):
    answer = api.get("/orders")
    if answer.status_code != 200:
        print(f"Call {call}: status {answer.status_code}")
        print("  The API says:  ", answer.json()["detail"])
        print("  Retry-After:   ", answer.headers["Retry-After"], "seconds")
        break
else:
    print("All calls passed.")

It is now Tuesday 2026-06-09 07:00:00 in Arden.
Call 13: status 429
  The API says:   Too many requests. This API allows 12 calls in 10 seconds. Try again in 10 seconds.
  Retry-After:    10 seconds


Status **429** means "too many requests". The API is not broken. It is asking you to slow down, and the `Retry-After` header says for how long.

## Step 3: A server error

The other common "no" is a **500**: something failed inside the order system. It usually passes in a moment. This fake API lets you plan such a failure.

In [3]:
set_morning("2026-06-09")
api.post("/admin/faults", json={"kind": "error_500", "count": 2})     # the next 2 calls will fail

try:
    read_orders(updated_since="2026-06-08T03:48:53Z")
except httpx.HTTPStatusError as error:
    print("The load crashed:", error.response.status_code, error.response.json()["detail"])

It is now Tuesday 2026-06-09 07:00:00 in Arden.
The load crashed: 500 The order system had an internal error. Try again.


The plain loop gives up at the first failure. One hiccup in the order system, and the manager has no report.

## Step 4: Wait, and try again

Build it by hand once. Read every line: this is the whole idea.

In [4]:
def get_page(params):
    """Ask for one page. If the API says 'not now', wait and ask again. At most 5 tries."""
    backoff = [1, 2, 4, 8]                       # seconds to wait after a server error
    for attempt in range(1, 6):
        answer = api.get("/orders", params=params)
        if answer.status_code == 200:
            return answer.json()
        if answer.status_code == 429:
            wait = float(answer.headers["Retry-After"])     # the API told us how long
        elif answer.status_code >= 500:
            wait = backoff[min(attempt, 4) - 1]             # each wait is longer than the last
        else:
            raise RuntimeError(f"Status {answer.status_code}: a retry cannot fix this.")
        if attempt < 5:
            print(f"  try {attempt}: status {answer.status_code}. Waiting {wait:.0f} seconds.")
            time.sleep(wait)
    raise RuntimeError("Still failing after 5 tries. Giving up.")


def read_changed(updated_since=None):
    """The loop from notebook 1, now with get_page inside."""
    orders, cursor = [], None
    while True:
        params = {key: value for key, value in [("updated_since", updated_since), ("cursor", cursor)] if value}
        body = get_page(params)
        orders += body["data"]
        cursor = body["next_cursor"]
        if cursor is None:
            return orders

In [5]:
api.post("/admin/faults", json={"kind": "error_500", "count": 2})     # the same failure as before

orders = read_changed("2026-06-08T03:48:53Z")
print(f"Done. {len(orders)} orders.")

  try 1: status 500. Waiting 1 seconds.


  try 2: status 500. Waiting 2 seconds.


Done. 29 orders.


The same two failures, and this time the load finished. It waited 1 second, then 2.

Waiting longer each time is called **backoff**. A source that is struggling needs time. Asking again at once makes it worse.

## Step 5: The morning milestone 1 could not load

Now a real rate limit. On the morning of Friday 12 June, a full load needs 14 calls. The API allows 12. Milestone 1 could not load this morning at all.

This cell really waits, about ten seconds.

In [6]:
set_morning("2026-06-12")

started = time.time()
orders = read_changed()             # no watermark: read everything
print(f"Done. {len(orders)} orders in {time.time() - started:.0f} seconds.")

It is now Friday 2026-06-12 07:00:00 in Arden.
  try 1: status 429. Waiting 10 seconds.


Done. 347 orders in 10 seconds.


The 13th call was refused. The load waited as long as the API asked, and carried on. Nothing was lost.

## Step 6: Always cap the tries

What if the order system is really down? The production code has the same logic as your `get_page`. It takes its "sleep" as an argument, so here it can **record** the waits and not really wait.

In [7]:
from daily_orders.extract import ExtractFailed, fetch_changed_orders

set_morning("2026-06-09")
api.post("/admin/faults", json={"kind": "error_500", "count": 99})    # down for a long time

waits = []
try:
    fetch_changed_orders(api, "2026-06-08T03:48:53Z", sleep=waits.append)
except ExtractFailed as error:
    print("The load stopped:", error)
print("Waits it asked for, in seconds:", waits)

_ = api.post("/admin/faults", json={"kind": "error_500", "count": 0})     # the order system is back

It is now Tuesday 2026-06-09 07:00:00 in Arden.
The load stopped: The API still answered 500 after 5 tries. Nothing was loaded.
Waits it asked for, in seconds: [1, 2, 4, 8]


Five tries, four waits, then a clear error. A job that retries for ever hides a broken source. Nobody is told, and the report is silently missing.

## Step 7: What not to retry

Some errors are **your** mistake. A retry sends the same mistake again.

In [8]:
answer = api.get("/orders", params={"updated_since": "yesterday"})
print(answer.status_code, answer.json()["detail"])

400 updated_since must look like 2026-06-08T12:00:00Z


In [9]:
show_doc("LLD.md", "3. Extract (`extract.py`)")

**From `docs/LLD.md`**

> ## 3. Extract (`extract.py`)
> 
> *(There is a diagram here. Open `docs/LLD.md` to see it.)*
> 
> | Function | Returns |
> |---|---|
> | `fetch_changed_orders(client, updated_since, sleep)` | `Extract(orders, pages, api_calls, retries, waited_seconds)`. `updated_since=None` reads everything |
> | `get_page(client, params, extract, sleep)` | One page, after as many tries as it needs |
> 
> `sleep` is passed in, so a test can count the waits without waiting.

| Status | Meaning | What to do |
|---|---|---|
| 429 | Too many requests | Wait as long as `Retry-After` says, then retry |
| 500 to 504 | The server failed | Retry with backoff |
| 400, 404 | Your request is wrong | Do not retry. Fix the request |

## Two new words

| Word | Meaning | Here |
|---|---|---|
| **Rate limit** | "A rate limit caps how many calls you may make in a period." | 12 calls in 10 seconds |
| **Retry with backoff** | Try a failed call again after a wait. "Exponential backoff means each wait limit is larger than the last." | Waits of 1, 2, 4, 8 seconds, and at most 5 tries |

The definitions are from the D8Loop chapter [Extraction: Pulling Data Reliably](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/extraction-pulling-data-reliably).

## What you now know

- 429 and 500 mean "not now". A good load waits and asks again.
- Backoff makes each wait longer than the last. The tries are always capped.
- A 400 is your own mistake. A retry cannot fix it.
- The morning that killed milestone 1 now loads, about ten seconds late.

**Say it in an interview:** "My extract retries 429 using the Retry-After header, and 5xx with exponential backoff: 1, 2, 4, 8 seconds, capped at five tries. It never retries a 400. After the last try it fails loudly and loads nothing, so a broken source is seen and not hidden."

A retry has a side effect. If a call fails after the API sent the rows, the retry brings them a second time. Keep that in mind. Next: [notebook 4](04_land_it_raw.ipynb) gives the orders a proper home.